In [1]:
from __future__ import annotations

import os
import json
import pandas as pd

import config

from utils import (
    prepare_train_val_test_split,
    clip_to_valid_range,
)
from utils import load_or_prepare_data
from tech_env_module import TechnicalEnv
from walkforward_module import walk_forward_validation, compare_algorithms


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:

prices, features = load_or_prepare_data()

(train_p, train_f), (val_p, val_f), (test_p, test_f), scaler = prepare_train_val_test_split(
    prices,
    features,
    asset_name=None,
    train_ratio=0.70,
    val_ratio=0.15,
    lookback_long=config.DATA_CONFIG['lookback_short'],
    clip_pre_ipo=True,
)

# Build multi‑asset environments; features are already normalised
train_env = TechnicalEnv(train_p, train_f, **config.ENV_CONFIG)
val_env = TechnicalEnv(val_p, val_f, **config.ENV_CONFIG)
test_env = TechnicalEnv(test_p, test_f, **config.ENV_CONFIG)

# Perform walk‑forward validation on the full multi‑asset series
wf_results = walk_forward_validation(
    prices,
    features,
    asset_name=None,
    env_config=config.ENV_CONFIG,
    model_config=config.MODEL_CONFIG,
    wf_config=config.WALKFORWARD_CONFIG,
    algorithm=config.MODEL_CONFIG['algorithm'],
    clip_pre_ipo=True,
)

print("\nMulti‑asset walk‑forward evaluation results:")
print(json.dumps({k: v if not isinstance(v, pd.DataFrame) else v.to_dict(orient='list')
                    for k, v in wf_results.items()}, indent=2, default=str))


  ✓ Input validation passed
    Periods: 389
    Assets: 10
    Features: 240
    Feature mean: 0.000 (expected ~0)
    Feature std:  1.000 (expected ~1)
  ✓ Input validation passed
    Periods: 83
    Assets: 10
    Features: 240
    Feature mean: 0.533 (expected ~0)
    Feature std:  0.949 (expected ~1)
  ✓ Input validation passed
    Periods: 85
    Assets: 10
    Features: 240
    Feature mean: 0.492 (expected ~0)
    Feature std:  1.173 (expected ~1)
WALK-FORWARD VALIDATION: MULTI-ASSET - SAC

Clipping pre-IPO backfill...

Window Configuration:
  Train: 156 weeks (~3.0 years)
  Val:   52 weeks (~1.0 years)
  Test:  26 weeks (~0.5 years)
  Gap:   4 weeks
  Step:  26 weeks

────────────────────────────────────────────────────────────────────────────────
Window 1:
  Train: 2020-07-01 to 2022-02-21
  Val:   2022-02-23 to 2022-08-22
  Gap:   2022-08-24 to 2022-09-05
  Test:  2022-09-07 to 2022-12-05
  Preparing data (normalizing on train only)...
  ✓ Input validation passed
    Periods

KeyboardInterrupt: 

In [2]:
# 1. Reload config to get new values
import importlib
importlib.reload(config)

print(f"Composite Weights:")
print(f"  Return weight:    {config.ENV_CONFIG['composite_return_weight']}")
print(f"  Turnover weight:  {config.ENV_CONFIG['composite_turnover_weight']}")
print(f"  Balance ratio:    {config.ENV_CONFIG['composite_return_weight']/config.ENV_CONFIG['composite_turnover_weight']:.2f}:1")
print(f"\nOther changes:")
print(f"  Reward lookback:  {config.ENV_CONFIG['reward_lookback']} weeks")
print(f"  Transaction cost: {config.ENV_CONFIG['transaction_cost']*10000:.0f} bps")
print(f"  Batch size:       {config.MODEL_CONFIG['batch_size']}")
print("="*80)

# 2. Retrain with fixed config
from utils import load_or_prepare_data, prepare_train_val_test_split
from tech_env_module import TechnicalEnv
from stable_baselines3 import SAC
import time

prices, features = load_or_prepare_data()
test_asset = 'NVDA'

print(f"\nRetraining {test_asset} with FIXED config...")

# Extract asset
asset_prices = prices[test_asset]
asset_features = features[[c for c in features.columns if c.startswith(f'{test_asset}_')]]
asset_features.columns = [c.replace(f'{test_asset}_', '') for c in asset_features.columns]

# Split
(train_p, train_f), (val_p, val_f), (test_p, test_f), scaler = prepare_train_val_test_split(
    asset_prices, asset_features, test_asset
)

# Train
train_env = TechnicalEnv(train_p, train_f, **config.ENV_CONFIG)
model = SAC('MlpPolicy', train_env, verbose=1, **config.get_model_hyperparams())

start = time.time()
model.learn(total_timesteps=config.MODEL_CONFIG['total_training_steps'])
elapsed = time.time() - start

print(f"\nTraining completed in {elapsed/60:.1f} minutes")

# 3. Evaluate
test_env = TechnicalEnv(test_p, test_f, **config.ENV_CONFIG)
obs, _ = test_env.reset()
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = test_env.step(action)
    done = terminated or truncated

metrics = test_env.get_portfolio_metrics()

print("\n" + "="*80)
print("IMPROVED RESULTS")
print("="*80)
print(f"  Sharpe:  {metrics['sharpe_ratio']:.3f}")
print(f"  Sortino: {metrics['sortino_ratio']:.3f}")
print(f"  Return:  {metrics['total_return']*100:.1f}%")
print(f"  Max DD:  {metrics['max_drawdown']*100:.1f}%")
print(f"  Trades:  {metrics['n_trades']} (vs 74 before)")
print("="*80)

# 4. Detailed trade analysis
from utils import print_trade_analysis
print_trade_analysis(test_env)

Composite Weights:
  Return weight:    3.0
  Turnover weight:  0.8
  Balance ratio:    3.75:1

Other changes:
  Reward lookback:  52 weeks
  Transaction cost: 30 bps
  Batch size:       512

Retraining NVDA with FIXED config...
  ✓ Input validation passed
    Periods: 350
    Assets: 1
    Features: 24
    Feature mean: 0.000 (expected ~0)
    Feature std:  1.000 (expected ~1)
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 349      |
|    ep_rew_mean     | 0.93     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 206      |
|    time_elapsed    | 6        |
|    total_timesteps | 1396     |
| train/             |          |
|    actor_loss      | -4.46    |
|    critic_loss     | 0.29     |
|    ent_coef        | 0.683    |
|    ent_coef_loss   | -0.599   |
|    learning_rate   | 0.0003   |
|    n_updates  

/Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/harlf_v4/tech_env_module.py:173: UserWarning: Features have high mean (0.590). Are they normalized? Expected mean ~0 for normalized features.
  warnings.warn(
